In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/vishleshmishra/harrypotter/HPBook7.txt
/kaggle/input/datasets/vishleshmishra/harrypotter/HPBook2.txt
/kaggle/input/datasets/vishleshmishra/harrypotter/HPBook6.txt
/kaggle/input/datasets/vishleshmishra/harrypotter/HPBook1.txt
/kaggle/input/datasets/vishleshmishra/harrypotter/HPBook3.txt
/kaggle/input/datasets/vishleshmishra/harrypotter/HPBook4.txt
/kaggle/input/datasets/vishleshmishra/harrypotter/HPBook5.txt


In [9]:
import pandas as pd 

Bookfile=[] 

# Loops through importing 7 HP text files - Book 1 creates table, Books 2-7 append to Book 1 table
for i in range(1,8): 
    Bookfile.append('HPBook'+str(i)+'.txt')
    FileLoc = "/kaggle/input/datasets/vishleshmishra/harrypotter/{}".format(Bookfile[i-1])
    if i == 1:
        df = pd.read_csv(FileLoc, sep="@")
    else:
        df2 = pd.read_csv(FileLoc, sep="@")
        df = pd.concat([df, df2])

In [10]:
df

,Text,Chapter,Book
0,"THE BOY WHO LIVED Mr. and Mrs. Dursley, of nu...",1,1
1,THE VANISHING GLASS Nearly ten years had pass...,2,1
2,THE LETTERS FROM NO ONE The escape of the Bra...,3,1
3,THE KEEPER OF THE KEYS BOOM. They knocked aga...,4,1
4,DIAGON ALLEY Harry woke early the next mornin...,5,1
...,...,...,...
32,"Harry remained kneeling at Snape's side, simpl...",33,7
33,"Finally, the truth. Lying with his face presse...",34,7
34,"He lay facedown, listening to the silence. He ...",35,7
35,He was flying facedown on the ground again. Th...,36,7


In [11]:
df1 = df.drop(['Chapter', 'Book'], axis =1)

In [28]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from tqdm import tqdm
import re
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Bookfile=[]
for i in range(1,8): 
    Bookfile.append('HPBook'+str(i)+'.txt')
    FileLoc = f"/kaggle/input/datasets/vishleshmishra/harrypotter/{Bookfile[i-1]}"
    if i == 1:
        df = pd.read_csv(FileLoc, sep="@")
    else:
        df2 = pd.read_csv(FileLoc, sep="@")
        df = pd.concat([df, df2])



Using device: cuda


In [46]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["Text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 2]

from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(sentence):
    return [w for w in sentence if w not in stop_words]

sentences = [remove_stopwords(s) for s in sentences]

sentences = df["clean_text"].apply(lambda x: x.split()).tolist()
sentences = [s for s in sentences if len(s) > 2]

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
words = [w for s in sentences for w in s]
word_counts = Counter(words)

vocab = list(word_counts.keys())
vocab_size = len(vocab)

word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

total_words = sum(word_counts.values())

def subsample(sentence, t=1e-3):
    result = []
    for w in sentence:
        f = word_counts[w] / total_words
        p = 1 - np.sqrt(t / f)
        if random.random() > p:
            result.append(w)
    return result

sentences = [subsample(s) for s in sentences]

In [ ]:
window_size = 5
pairs = []

for sent in sentences:
    for i, target in enumerate(sent):
        for j in range(-window_size, window_size+1):
            if j != 0 and 0 <= i+j < len(sent):
                pairs.append((word2idx[target], word2idx[sent[i+j]]))

pairs = np.array(pairs)

freqs = np.array([word_counts[w] for w in vocab])
freqs = freqs ** 0.75
freqs = freqs / np.sum(freqs)

In [ ]:
#model
embedding_dim = 300

class Word2Vec(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, embedding_dim)
        self.out_embed = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, target, pos_context, neg_context):
        v = self.in_embed(target)                  # (batch, dim)
        pos = self.out_embed(pos_context)          # (batch, dim)
        neg = self.out_embed(neg_context)          # (batch, k, dim)

        pos_score = torch.sum(v * pos, dim=1)      # (batch)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-10)

        neg_score = torch.bmm(neg, v.unsqueeze(2)).squeeze()  # (batch, k)
        neg_loss = -torch.sum(torch.log(torch.sigmoid(-neg_score) + 1e-10), dim=1)

        return torch.mean(pos_loss + neg_loss)

model = Word2Vec(vocab_size, embedding_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.003)

In [47]:
# ===================== TRAINING =====================
epochs = 10
batch_size = 1024
k = 5   # negative samples

pairs_tensor = torch.tensor(pairs)

for epoch in range(epochs):
    total_loss = 0
    
    perm = torch.randperm(len(pairs_tensor))
    pairs_tensor = pairs_tensor[perm]

    for i in tqdm(range(0, len(pairs_tensor), batch_size), desc=f"Epoch {epoch+1}"):

        batch = pairs_tensor[i:i+batch_size]
        
        target = batch[:,0].to(device)
        context = batch[:,1].to(device)

        # negative sampling (vectorized)
        neg_samples = np.random.choice(vocab_size, size=(len(batch), k), p=freqs)
        neg_samples = torch.tensor(neg_samples).to(device)

        loss = model(target, context, neg_samples)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss}")

embeddings = model.in_embed.weight.data.cpu().numpy()

print(most_similar("harry"))

Epoch 1: 100%|██████████| 7333/7333 [00:53<00:00, 137.00it/s]


Epoch 1, Loss: 90911.49629163742


Epoch 2: 100%|██████████| 7333/7333 [00:53<00:00, 137.12it/s]


Epoch 2, Loss: 28595.53323864937


Epoch 3: 100%|██████████| 7333/7333 [00:53<00:00, 137.29it/s]


...

Epoch 8, Loss: 17612.0744035244


Epoch 9: 100%|██████████| 7333/7333 [00:53<00:00, 137.03it/s]


Epoch 9, Loss: 17371.098774194717


Epoch 10: 100%|██████████| 7333/7333 [00:53<00:00, 137.03it/s]


Epoch 10, Loss: 17184.933244228363
['he', 'ron', 'as', 'hermione', 'said']


In [65]:
from sklearn.metrics.pairwise import cosine_similarity

def most_similar(word, top_k=10):
    if word not in word2idx:
        return "Word not found"
    
    vec = embeddings[word2idx[word]].reshape(1, -1)
    sims = cosine_similarity(vec, embeddings)[0]
    
    top_idx = sims.argsort()[-top_k-1:-1][::-1]
    return [idx2word[i] for i in top_idx]

In [86]:
print(most_similar("phoenix"))

['feather', 'order', 'fawkes', 'veritaserum', 'holly', 'newcomers', 'spurted', 'writer', 'hogwartsany', 'willow']
